# Stress Granule Segmentation — Depthwise U-Net vs Attention U-Net
### Group 7 — Rahul's part

This notebook trains and evaluates two lightweight U-Net variants on the
pseudo-labelled stress-granule dataset, and compares both against the
reference YOLO11n+U-Net numbers given in the instructions file.

**Before running:** upload `DATA_FOR_TEAM.zip` to this Colab session (left
sidebar → Files → upload, or mount Drive) and unzip it so you have a
`DATA_FOR_TEAM/` folder next to this notebook. Set Runtime → Change runtime
type → GPU first.

Run the cells top to bottom. Cells marked **CONFIRM** print something you
should check before moving on — the instructions explicitly say not to
assume key names or counts.


## 0. Setup

In [ ]:
!pip install -q torch torchvision scikit-image scipy pandas matplotlib

import os, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from skimage.measure import label as cc_label
from scipy.ndimage import distance_transform_edt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(0)


In [ ]:
# If your zip isn't unzipped yet, uncomment:
# !unzip -q DATA_FOR_TEAM.zip -d .

DATA_DIR = "DATA_FOR_TEAM"
assert os.path.isdir(DATA_DIR), f"Can't find {DATA_DIR} — upload + unzip it first."
print(os.listdir(DATA_DIR))


## 1. Inspect the data (CONFIRM before continuing)

Do not assume key names — print them and adjust `FRAMES_KEY` / `MASKS_KEY`
below if they differ from `reg` / `granules`.


In [ ]:
for f in ["v1_pre", "v2_pre", "v1_pl", "v2_pl"]:
    z = np.load(os.path.join(DATA_DIR, f + ".npz"))
    print(f, {k: (z[k].shape, z[k].dtype) for k in z.files})

split = json.load(open(os.path.join(DATA_DIR, "split.json")))
print(split.keys())
print("train v2 frames:", len(split["train"]["v2"]))
print("val   v2 frames:", len(split["val"]["v2"]))
print("test  v1 frames:", len(split["test"]["v1"]))


In [ ]:
# EDIT THESE two lines once you've confirmed the real key names from the cell above.
FRAMES_KEY = "reg"        # e.g. "reg" — the raw/registered frame array
MASKS_KEY  = "granules"   # e.g. "granules" — the integer instance-label mask

v2_pre = np.load(os.path.join(DATA_DIR, "v2_pre.npz"))[FRAMES_KEY]
v2_pl  = np.load(os.path.join(DATA_DIR, "v2_pl.npz"))[MASKS_KEY]
v1_pre = np.load(os.path.join(DATA_DIR, "v1_pre.npz"))[FRAMES_KEY]
v1_pl  = np.load(os.path.join(DATA_DIR, "v1_pl.npz"))[MASKS_KEY]

print("v2_pre", v2_pre.shape, v2_pre.dtype, "min/max", v2_pre.min(), v2_pre.max())
print("v2_pl ", v2_pl.shape, v2_pl.dtype, "max id", v2_pl.max())
print("v1_pre", v1_pre.shape)
print("v1_pl ", v1_pl.shape)

# split.json frame numbers are 1-indexed per the instructions; confirm vs array indexing
train_idx = [i - 1 for i in split["train"]["v2"]]
val_idx   = [i - 1 for i in split["val"]["v2"]]
test_idx  = [i - 1 for i in split["test"]["v1"]]
print(len(train_idx), len(val_idx), len(test_idx))


In [ ]:
# Quick visual sanity check: one frame + its mask
import matplotlib.pyplot as plt
t = train_idx[0]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(v2_pre[t], cmap="gray"); ax[0].set_title(f"frame {t} (raw)")
ax[1].imshow(v2_pl[t] > 0, cmap="gray"); ax[1].set_title(f"frame {t} (binary mask)")
for a in ax: a.axis("off")
plt.show()
print("granules in this frame:", v2_pl[t].max())


## 2. Tile the frames into 256x256 patches

Reference set used 256px tiles with ~20% empty (no-granule) tiles so the
model learns "no object." We build tiles for train/val from video 2. Test
stays as full 1024x1024 frames (evaluated separately in section 6).


In [ ]:
TILE = 256
STRIDE = 256  # non-overlapping; increase overlap later if you want more tiles

def make_tiles(frames, masks, indices, tile=TILE, stride=STRIDE, keep_empty_frac=0.2, seed=0):
    rng = random.Random(seed)
    pos_tiles, neg_tiles = [], []
    H, W = frames.shape[1:]
    for idx in indices:
        frame = frames[idx]
        mask = (masks[idx] > 0).astype(np.float32)
        for y in range(0, H - tile + 1, stride):
            for x in range(0, W - tile + 1, stride):
                fp = frame[y:y+tile, x:x+tile]
                mp = mask[y:y+tile, x:x+tile]
                if mp.sum() > 0:
                    pos_tiles.append((fp.copy(), mp.copy()))
                else:
                    neg_tiles.append((fp.copy(), mp.copy()))
    n_keep_neg = int(len(pos_tiles) * keep_empty_frac / (1 - keep_empty_frac))
    rng.shuffle(neg_tiles)
    kept_neg = neg_tiles[:n_keep_neg]
    tiles = pos_tiles + kept_neg
    rng.shuffle(tiles)
    return tiles

train_tiles = make_tiles(v2_pre, v2_pl, train_idx)
val_tiles   = make_tiles(v2_pre, v2_pl, val_idx)
print("train tiles:", len(train_tiles), " val tiles:", len(val_tiles))


## 3. Dataset + DataLoader (with allowed augmentations)

In [ ]:
class GranuleTileDataset(Dataset):
    """tiles: list of (frame[H,W] float32 in [0,1], mask[H,W] float32 {0,1})"""
    def __init__(self, tiles, train=True, upscale=2):
        self.tiles = tiles
        self.train = train
        self.upscale = upscale  # matches the "2x upsampling" rule in the instructions

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, i):
        img, mask = self.tiles[i]
        img, mask = img.copy(), mask.copy()

        if self.train:
            # flips + 90-degree rotations
            if random.random() < 0.5:
                img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
            if random.random() < 0.5:
                img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
            k = random.choice([0, 1, 2, 3])
            img, mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy()
            # brightness/contrast jitter (mimics photobleaching) — NO hue/sat, it's grayscale
            if random.random() < 0.5:
                gain = random.uniform(0.8, 1.2)
                bias = random.uniform(-0.1, 0.1)
                img = np.clip(img * gain + bias, 0, 1)

        img_t = torch.from_numpy(img).float().unsqueeze(0)   # 1xHxW
        mask_t = torch.from_numpy(mask).float().unsqueeze(0)  # 1xHxW

        if self.upscale != 1:
            img_t = F.interpolate(img_t.unsqueeze(0), scale_factor=self.upscale,
                                   mode="bilinear", align_corners=False).squeeze(0)
            mask_t = F.interpolate(mask_t.unsqueeze(0), scale_factor=self.upscale,
                                    mode="nearest").squeeze(0)
        return img_t, mask_t


def make_loaders(upscale=2, batch_size=8):
    train_ds = GranuleTileDataset(train_tiles, train=True, upscale=upscale)
    val_ds   = GranuleTileDataset(val_tiles, train=False, upscale=upscale)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)
    val_dl   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()
xb, yb = next(iter(train_dl))
print(xb.shape, yb.shape, xb.min().item(), xb.max().item())


## 4. The two architectures

- **DepthwiseUNet**: every conv block uses depthwise-separable convolution
  (fewer parameters, cheaper — the MobileNet trick) instead of standard conv.
- **AttentionUNet**: standard conv blocks, but each skip connection passes
  through an attention gate that learns to suppress irrelevant (background)
  regions before merging into the decoder — meant to help with tiny objects.


In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))


class DoubleConv(nn.Module):
    """Standard double conv block, used inside AttentionUNet."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class DWDoubleConv(nn.Module):
    """Double depthwise-separable conv block, used inside DepthwiseUNet."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c1 = DepthwiseSeparableConv(in_ch, out_ch)
        self.c2 = DepthwiseSeparableConv(out_ch, out_ch)
    def forward(self, x):
        return self.c2(self.c1(x))


class AttentionGate(nn.Module):
    """g = coarser decoder gating signal, x = finer encoder skip connection.
    Learns a per-pixel soft mask over x that suppresses irrelevant regions."""
    def __init__(self, g_ch, x_ch, inter_ch):
        super().__init__()
        self.Wg = nn.Conv2d(g_ch, inter_ch, 1)
        self.Wx = nn.Conv2d(x_ch, inter_ch, 1)
        self.psi = nn.Conv2d(inter_ch, 1, 1)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
    def forward(self, g, x):
        g1 = self.Wg(g)
        x1 = self.Wx(x)
        if g1.shape[-2:] != x1.shape[-2:]:
            g1 = F.interpolate(g1, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        attn = self.sigmoid(self.psi(self.relu(g1 + x1)))
        return x * attn


class _BaseUNet(nn.Module):
    """Shared 4-level encoder/decoder skeleton; subclasses supply the conv block
    (plain vs depthwise) and whether skip connections pass through attention gates."""
    def __init__(self, conv_block, base=16, use_attention=False):
        super().__init__()
        C = base
        self.use_attention = use_attention
        self.enc1 = conv_block(1, C)
        self.enc2 = conv_block(C, C*2)
        self.enc3 = conv_block(C*2, C*4)
        self.enc4 = conv_block(C*4, C*8)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = conv_block(C*8, C*16)

        self.up4 = nn.ConvTranspose2d(C*16, C*8, 2, stride=2)
        self.up3 = nn.ConvTranspose2d(C*8, C*4, 2, stride=2)
        self.up2 = nn.ConvTranspose2d(C*4, C*2, 2, stride=2)
        self.up1 = nn.ConvTranspose2d(C*2, C, 2, stride=2)

        if use_attention:
            self.att4 = AttentionGate(C*8, C*8, C*4)
            self.att3 = AttentionGate(C*4, C*4, C*2)
            self.att2 = AttentionGate(C*2, C*2, C)
            self.att1 = AttentionGate(C, C, C//2 if C >= 2 else 1)

        self.dec4 = conv_block(C*16, C*8)
        self.dec3 = conv_block(C*8, C*4)
        self.dec2 = conv_block(C*4, C*2)
        self.dec1 = conv_block(C*2, C)

        self.out_conv = nn.Conv2d(C, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        s4 = self.att4(d4, e4) if self.use_attention else e4
        d4 = self.dec4(torch.cat([d4, s4], dim=1))

        d3 = self.up3(d4)
        s3 = self.att3(d3, e3) if self.use_attention else e3
        d3 = self.dec3(torch.cat([d3, s3], dim=1))

        d2 = self.up2(d3)
        s2 = self.att2(d2, e2) if self.use_attention else e2
        d2 = self.dec2(torch.cat([d2, s2], dim=1))

        d1 = self.up1(d2)
        s1 = self.att1(d1, e1) if self.use_attention else e1
        d1 = self.dec1(torch.cat([d1, s1], dim=1))

        return self.out_conv(d1)  # logits, 1xHxW


def DepthwiseUNet(base=16):
    return _BaseUNet(DWDoubleConv, base=base, use_attention=False)

def AttentionUNet(base=16):
    return _BaseUNet(DoubleConv, base=base, use_attention=True)


# sanity check
m = DepthwiseUNet(16).to(device)
out = m(xb.to(device))
print("DepthwiseUNet output:", out.shape, "params:", sum(p.numel() for p in m.parameters()))
m2 = AttentionUNet(16).to(device)
out2 = m2(xb.to(device))
print("AttentionUNet output:", out2.shape, "params:", sum(p.numel() for p in m2.parameters()))


## 5. Loss function

Dice + weighted BCE — the reference sweep found BCE weight 0.5 helped the
most, so that's the default starting point (edit `bce_weight` in the sweep
in section 7 if you want to test others).


In [ ]:
def combined_loss(logits, target, bce_weight=0.5, eps=1.0):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    probs = torch.sigmoid(logits)
    inter = (probs * target).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice_loss = 1 - ((2*inter + eps) / (union + eps)).mean()
    return dice_loss + bce_weight * bce

@torch.no_grad()
def dice_score(logits, target, eps=1.0):
    probs = (torch.sigmoid(logits) > 0.5).float()
    inter = (probs * target).sum(dim=(1,2,3))
    union = probs.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    return ((2*inter + eps) / (union + eps)).mean().item()


## 6. Training loop (with early stopping)

In [ ]:
def train_model(model, train_dl, val_dl, lr=1e-3, bce_weight=0.5,
                 max_epochs=40, patience=6, tag="model"):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_dice": []}
    best_dice = -1
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        t0 = time.time()
        train_losses = []
        for xb_, yb_ in train_dl:
            xb_, yb_ = xb_.to(device), yb_.to(device)
            opt.zero_grad()
            logits = model(xb_)
            loss = combined_loss(logits, yb_, bce_weight=bce_weight)
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses, val_dices = [], []
        with torch.no_grad():
            for xb_, yb_ in val_dl:
                xb_, yb_ = xb_.to(device), yb_.to(device)
                logits = model(xb_)
                val_losses.append(combined_loss(logits, yb_, bce_weight=bce_weight).item())
                val_dices.append(dice_score(logits, yb_))

        tr_loss = np.mean(train_losses)
        va_loss = np.mean(val_losses)
        va_dice = np.mean(val_dices)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_dice"].append(va_dice)

        print(f"[{tag}] epoch {epoch:02d} | train_loss {tr_loss:.4f} | "
              f"val_loss {va_loss:.4f} | val_dice {va_dice:.4f} | {time.time()-t0:.1f}s")

        if va_dice > best_dice:
            best_dice = va_dice
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{tag}] early stopping at epoch {epoch} (best val_dice {best_dice:.4f})")
                break

    model.load_state_dict(best_state)
    return model, history, best_dice


## 7. Baseline runs (both architectures, defaults)

Start here. ~30-45 min each on a Colab T4 depending on epoch count/tile count.


In [ ]:
set_seed(0)
dw_model, dw_history, dw_best_dice = train_model(
    DepthwiseUNet(base=16), train_dl, val_dl,
    lr=1e-3, bce_weight=0.5, max_epochs=40, patience=6, tag="DepthwiseUNet-baseline"
)


In [ ]:
set_seed(0)
att_model, att_history, att_best_dice = train_model(
    AttentionUNet(base=16), train_dl, val_dl,
    lr=1e-3, bce_weight=0.5, max_epochs=40, patience=6, tag="AttentionUNet-baseline"
)


## 8. Small hyperparameter sweep (val only — keep every run in a table)

Edit `SWEEP_CONFIGS` to add/remove combinations. Keep runs short (12 epochs,
like the reference sweep) — this is for comparing settings, not final training.


In [ ]:
SWEEP_CONFIGS = [
    {"arch": "depthwise", "base": 16, "lr": 1e-3, "bce_weight": 0.5, "upscale": 1},
    {"arch": "depthwise", "base": 16, "lr": 3e-3, "bce_weight": 0.5, "upscale": 1},
    {"arch": "depthwise", "base": 32, "lr": 1e-3, "bce_weight": 0.5, "upscale": 1},
    {"arch": "depthwise", "base": 16, "lr": 1e-3, "bce_weight": 0.5, "upscale": 2},
    {"arch": "attention", "base": 16, "lr": 1e-3, "bce_weight": 0.5, "upscale": 1},
    {"arch": "attention", "base": 16, "lr": 3e-3, "bce_weight": 0.5, "upscale": 1},
    {"arch": "attention", "base": 32, "lr": 1e-3, "bce_weight": 0.5, "upscale": 1},
    {"arch": "attention", "base": 16, "lr": 1e-3, "bce_weight": 0.5, "upscale": 2},
]

sweep_rows = []
for cfg in SWEEP_CONFIGS:
    tr_dl, va_dl = make_loaders(upscale=cfg["upscale"], batch_size=8)
    set_seed(0)
    arch_fn = DepthwiseUNet if cfg["arch"] == "depthwise" else AttentionUNet
    model = arch_fn(base=cfg["base"])
    tag = f"{cfg['arch']}_b{cfg['base']}_lr{cfg['lr']}_bce{cfg['bce_weight']}_up{cfg['upscale']}"
    t0 = time.time()
    _, hist, best_dice = train_model(
        model, tr_dl, va_dl, lr=cfg["lr"], bce_weight=cfg["bce_weight"],
        max_epochs=12, patience=4, tag=tag
    )
    elapsed = time.time() - t0
    sweep_rows.append({**cfg, "val_dice": best_dice, "train_time_s": round(elapsed, 1)})

import pandas as pd
sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("sweep_results.csv", index=False)
sweep_df.sort_values("val_dice", ascending=False)


## 9. Final runs — best config per architecture

Read the best row per architecture from `sweep_df` above, plug the settings
in here, and train properly to convergence.


In [ ]:
# EDIT these based on what the sweep table showed you
BEST_DW  = {"base": 16, "lr": 1e-3, "bce_weight": 0.5, "upscale": 1}
BEST_ATT = {"base": 16, "lr": 1e-3, "bce_weight": 0.5, "upscale": 1}

tr_dl, va_dl = make_loaders(upscale=BEST_DW["upscale"], batch_size=8)
set_seed(0)
final_dw_model, final_dw_history, final_dw_dice = train_model(
    DepthwiseUNet(base=BEST_DW["base"]), tr_dl, va_dl,
    lr=BEST_DW["lr"], bce_weight=BEST_DW["bce_weight"],
    max_epochs=60, patience=8, tag="DepthwiseUNet-final"
)
torch.save(final_dw_model.state_dict(), "depthwise_unet_final.pt")


In [ ]:
tr_dl, va_dl = make_loaders(upscale=BEST_ATT["upscale"], batch_size=8)
set_seed(0)
final_att_model, final_att_history, final_att_dice = train_model(
    AttentionUNet(base=BEST_ATT["base"]), tr_dl, va_dl,
    lr=BEST_ATT["lr"], bce_weight=BEST_ATT["bce_weight"],
    max_epochs=60, patience=8, tag="AttentionUNet-final"
)
torch.save(final_att_model.state_dict(), "attention_unet_final.pt")


In [ ]:
# Training/validation curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in [("DepthwiseUNet", final_dw_history), ("AttentionUNet", final_att_history)]:
    axes[0].plot(hist["train_loss"], label=f"{name} train")
    axes[0].plot(hist["val_loss"], linestyle="--", label=f"{name} val")
    axes[1].plot(hist["val_dice"], label=name)
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")
axes[1].set_title("Val Dice"); axes[1].legend(); axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()


## 10. Evaluate on test (video 1, full 1024x1024 frames — only now)

This is the one point where the test set gets touched. For each test frame:
run the model tile-by-tile (or on the whole frame if your GPU memory allows),
reassemble a full-frame binary prediction, then run connected-component
labeling to split it into instances for the instance-level metrics.


In [ ]:
def predict_full_frame(model, frame, upscale=1, tile=256, device=device):
    """Run model over a full 1024x1024 frame via non-overlapping tiles, reassemble."""
    model.eval()
    H, W = frame.shape
    pred = np.zeros((H, W), dtype=np.float32)
    with torch.no_grad():
        for y in range(0, H, tile):
            for x in range(0, W, tile):
                patch = frame[y:y+tile, x:x+tile]
                ph, pw = patch.shape
                # pad if at the edge and frame isn't an exact multiple of tile
                pad_h, pad_w = tile - ph, tile - pw
                if pad_h > 0 or pad_w > 0:
                    patch = np.pad(patch, ((0, pad_h), (0, pad_w)), mode="reflect")
                t = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0)
                if upscale != 1:
                    t = F.interpolate(t, scale_factor=upscale, mode="bilinear", align_corners=False)
                t = t.to(device)
                logits = model(t)
                probs = torch.sigmoid(logits)
                if upscale != 1:
                    probs = F.interpolate(probs, size=(tile, tile), mode="bilinear", align_corners=False)
                probs = probs.squeeze().cpu().numpy()
                pred[y:y+ph, x:x+pw] = probs[:ph, :pw]
    return pred  # float probability map, same size as input frame


In [ ]:
def match_instances(pred_label, gt_label, iou_thresh=0.5):
    """Greedy IoU matching between predicted and ground-truth instance maps.
    Returns list of (pred_id, gt_id, iou) matches plus unmatched pred/gt ids."""
    pred_ids = [i for i in np.unique(pred_label) if i != 0]
    gt_ids = [i for i in np.unique(gt_label) if i != 0]

    ious = np.zeros((len(pred_ids), len(gt_ids)))
    pred_masks = {pid: (pred_label == pid) for pid in pred_ids}
    gt_masks = {gid: (gt_label == gid) for gid in gt_ids}

    for i, pid in enumerate(pred_ids):
        pm = pred_masks[pid]
        # only check gt instances whose bbox roughly overlaps — fine to brute force at this object count
        for j, gid in enumerate(gt_ids):
            gm = gt_masks[gid]
            inter = np.logical_and(pm, gm).sum()
            if inter == 0:
                continue
            union = np.logical_or(pm, gm).sum()
            ious[i, j] = inter / union

    matches = []
    used_pred, used_gt = set(), set()
    flat = [(ious[i, j], i, j) for i in range(len(pred_ids)) for j in range(len(gt_ids)) if ious[i, j] >= iou_thresh]
    flat.sort(reverse=True)
    for iou, i, j in flat:
        if i in used_pred or j in used_gt:
            continue
        used_pred.add(i); used_gt.add(j)
        matches.append((pred_ids[i], gt_ids[j], iou))

    unmatched_pred = [pred_ids[i] for i in range(len(pred_ids)) if i not in used_pred]
    unmatched_gt = [gt_ids[j] for j in range(len(gt_ids)) if j not in used_gt]
    return matches, unmatched_pred, unmatched_gt, pred_masks, gt_masks


def aji(pred_label, gt_label):
    """Aggregated Jaccard Index."""
    gt_ids = [i for i in np.unique(gt_label) if i != 0]
    pred_ids = set(i for i in np.unique(pred_label) if i != 0)
    used_pred = set()
    inter_sum, union_sum = 0, 0
    for gid in gt_ids:
        gm = gt_label == gid
        best_iou, best_pid, best_inter, best_union = 0, None, 0, gm.sum()
        for pid in pred_ids:
            pm = pred_label == pid
            inter = np.logical_and(gm, pm).sum()
            if inter == 0:
                continue
            union = np.logical_or(gm, pm).sum()
            iou = inter / union
            if iou > best_iou:
                best_iou, best_pid, best_inter, best_union = iou, pid, inter, union
        if best_pid is not None:
            inter_sum += best_inter
            union_sum += best_union
            used_pred.add(best_pid)
        else:
            union_sum += gm.sum()
    for pid in pred_ids - used_pred:
        union_sum += (pred_label == pid).sum()
    return inter_sum / union_sum if union_sum > 0 else 0.0


def centroid(mask):
    ys, xs = np.nonzero(mask)
    return np.array([ys.mean(), xs.mean()])


def evaluate_test_frames(model, frames, gt_masks_instance, indices, upscale=1, thresh=0.5):
    all_rows = []
    pixel_inter, pixel_union = 0, 0
    pixel_tp, pixel_fp, pixel_fn = 0, 0, 0
    total_time = 0.0

    for idx in indices:
        frame = frames[idx]
        gt_label = gt_masks_instance[idx].astype(np.int32)

        t0 = time.time()
        prob = predict_full_frame(model, frame, upscale=upscale)
        total_time += time.time() - t0

        pred_bin = (prob > thresh).astype(np.uint8)
        pred_label = cc_label(pred_bin).astype(np.int32)

        gt_bin = (gt_label > 0).astype(np.uint8)
        pixel_inter += np.logical_and(pred_bin, gt_bin).sum()
        pixel_union += np.logical_or(pred_bin, gt_bin).sum()
        pixel_tp += np.logical_and(pred_bin, gt_bin).sum()
        pixel_fp += np.logical_and(pred_bin, 1 - gt_bin).sum()
        pixel_fn += np.logical_and(1 - pred_bin, gt_bin).sum()

        matches, unmatched_pred, unmatched_gt, pred_masks, gt_masks_d = match_instances(pred_label, gt_label)

        n_tp = len(matches)
        n_fp = len(unmatched_pred)
        n_fn = len(unmatched_gt)

        ious_matched = [m[2] for m in matches]
        dices_matched = [2*m[2]/(1+m[2]) for m in matches]  # Dice from IoU: 2*IoU/(1+IoU)

        loc_errors = []
        point_f1_hits = 0
        for pid, gid, iou in matches:
            c_pred = centroid(pred_masks[pid])
            c_gt = centroid(gt_masks_d[gid])
            d = np.linalg.norm(c_pred - c_gt)
            loc_errors.append(d)
            if d <= 3.0:
                point_f1_hits += 1

        this_aji = aji(pred_label, gt_label)

        all_rows.append({
            "frame": idx, "tp": n_tp, "fp": n_fp, "fn": n_fn,
            "mean_iou": np.mean(ious_matched) if ious_matched else 0,
            "mean_dice": np.mean(dices_matched) if dices_matched else 0,
            "mean_loc_err": np.mean(loc_errors) if loc_errors else np.nan,
            "point_f1_hits": point_f1_hits,
            "aji": this_aji,
        })

    df = pd.DataFrame(all_rows)
    total_tp, total_fp, total_fn = df.tp.sum(), df.fp.sum(), df.fn.sum()
    precision = total_tp / (total_tp + total_fp) if (total_tp+total_fp) else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp+total_fn) else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    point_f1 = 2*df.point_f1_hits.sum()/(total_tp+total_fn+total_tp) if total_tp else 0

    summary = {
        "pixel_iou": pixel_inter/pixel_union if pixel_union else 0,
        "pixel_dice": 2*pixel_tp/(2*pixel_tp+pixel_fp+pixel_fn) if (2*pixel_tp+pixel_fp+pixel_fn) else 0,
        "instance_precision": precision,
        "instance_recall": recall,
        "instance_f1": f1,
        "mean_matched_iou": df.mean_iou[df.tp>0].mean() if (df.tp>0).any() else 0,
        "mean_matched_dice": df.mean_dice[df.tp>0].mean() if (df.tp>0).any() else 0,
        "mean_localisation_error_px": df.mean_loc_err.mean(),
        "point_f1_3px": point_f1,
        "aji": df.aji.mean(),
        "fps": len(indices) / total_time if total_time > 0 else 0,
    }
    return summary, df


In [ ]:
dw_summary, dw_perframe = evaluate_test_frames(
    final_dw_model, v1_pre, v1_pl, test_idx, upscale=BEST_DW["upscale"]
)
att_summary, att_perframe = evaluate_test_frames(
    final_att_model, v1_pre, v1_pl, test_idx, upscale=BEST_ATT["upscale"]
)

reference = {
    "pixel_iou": 0.600, "pixel_dice": 0.746,
    "instance_precision": 0.693, "instance_recall": 0.872, "instance_f1": 0.771,
    "mean_matched_iou": 0.870, "mean_matched_dice": 0.927,
    "mean_localisation_error_px": 0.43, "point_f1_3px": 0.759,
    "aji": 0.563, "fps": 5.5,
}

results_table = pd.DataFrame({
    "Reference (YOLO11n+UNet)": reference,
    "DepthwiseUNet": dw_summary,
    "AttentionUNet": att_summary,
}).round(3)
results_table.to_csv("test_results_comparison.csv")
results_table


## 11. Visual examples (3–5 test frames, ground truth vs. prediction overlaid)


In [ ]:
def overlay_preview(frame, gt_label, pred_prob, thresh=0.5):
    pred_bin = pred_prob > thresh
    gt_bin = gt_label > 0
    rgb = np.stack([frame, frame, frame], axis=-1)
    rgb = np.clip(rgb, 0, 1)
    overlay = rgb.copy()
    overlay[gt_bin] = [1, 0, 0]       # red = ground truth
    overlay[pred_bin] = overlay[pred_bin] * 0.3 + np.array([0, 1, 0]) * 0.7  # green = prediction
    overlay[np.logical_and(gt_bin, pred_bin)] = [1, 1, 0]  # yellow = agreement
    return overlay

sample_idx = test_idx[:5]
fig, axes = plt.subplots(1, len(sample_idx), figsize=(4*len(sample_idx), 4))
for ax, idx in zip(axes, sample_idx):
    frame = v1_pre[idx]
    gt = v1_pl[idx]
    prob = predict_full_frame(final_dw_model, frame, upscale=BEST_DW["upscale"])
    ax.imshow(overlay_preview(frame, gt, prob))
    ax.set_title(f"frame {idx}")
    ax.axis("off")
plt.suptitle("Red=ground truth only, Green=prediction only, Yellow=agreement (DepthwiseUNet)")
plt.tight_layout()
plt.savefig("test_examples_depthwise.png", dpi=150)
plt.show()


## 12. (Optional, if time allows) Zero-shot Cellpose / SAM baselines

Only do this after everything above is done and saved. Run Cellpose and SAM's
automatic mask generator on the same test frames, tuned to the granule scale
(diameter ~3-10px for Cellpose; small point-grid spacing for SAM), and report
the same metrics using `evaluate_test_frames`-style matching. State whatever
settings you used.


In [ ]:
# !pip install -q cellpose
# from cellpose import models as cp_models
# cp_model = cp_models.Cellpose(gpu=True, model_type='cyto')
# masks, flows, styles, diams = cp_model.eval(v1_pre[test_idx[0]], diameter=8, channels=[0,0])
# (then run the same connected-component / matching code above on `masks`)

# SAM: use the automatic mask generator from segment-anything, with a dense point grid
# and small min_mask_region_area, tuned for ~3-10px objects.
print("Fill in if time allows — not required for the core deliverable.")


## 13. Written summary (fill in after running everything)

- What you trained: DepthwiseUNet and AttentionUNet, base channels = ?, final
  config = ?
- What worked / what didn't (compare the two architectures to each other,
  and to the reference numbers above)
- **Limitations to state honestly:**
  - Labels are pseudo-labels from a classical method, not human-verified
    ground truth — high agreement means "reproduces the classical method,"
    not "is medically correct." Confirm against hand-labelled tiles once
    available.
  - Touching/overlapping granules can get merged into one blob by
    connected-component labeling, unlike a detector-first pipeline
    (Mask R-CNN/YOLO) which separates instances natively — this is a real
    structural weak point of a pure-segmentation approach.
  - Domain shift: test is a different video (video 1) with a different
    frame interval than train/val (video 2).
  - Only 2 source videos total.
  - State your actual GPU (e.g. Colab T4) for the fps number, since
    throughput is hardware-dependent.
